# Step 1 — Load and inspect the raw dataset

Read `LJ_Dataset_ORIG.csv` into `df`, then inspect shape, data types, and row count.

This first check confirms the dataset loaded correctly before any cleaning is applied.

In [11]:
import pandas as pd

df = pd.read_csv('data/raw/LJ_Dataset_ORIG.csv')
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nNumber of rows:", len(df))

Shape: (1421, 29)

Data Types:
G               int64
Date              str
Age               str
Tm                str
Unnamed: 5        str
               ...   
TOV             int64
PF              int64
PTS             int64
GmSc          float64
+/-           float64
Length: 29, dtype: object

Number of rows: 1421


# Step 2 — Fix the `Date` feature

Convert `Date` to datetime using `errors='coerce'` so invalid values become `NaT`.

Why this is needed for training: model pipelines and chronological splits require a valid time type, not plain text.

In [12]:
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    print(df['Date'].dtype)
    display(df[['Date']].head())
else:
    print("'Date' column not found.")

datetime64[us]


,Date
0,2003-10-29
1,2003-10-30
2,2003-11-01
3,2003-11-05
4,2003-11-07


# Step 3 — Standardize Home/Away indicator

Rename `Unnamed: 5` to `Home` and convert values to binary: `@ -> 0` (away), otherwise `1` (home).

Why this is needed for training: binary numeric encoding is model-ready and keeps home-court context in a consistent format.

In [13]:
# Rename Unnamed: 5 to Home
df.rename(columns={'Unnamed: 5': 'Home'}, inplace=True)

# Convert to binary: @ means away (0), NaN means home (1)
df['Home'] = df['Home'].apply(lambda x: 0 if x == '@' else 1)

print(df[['Home']].value_counts())
print(df.head())

Home
1       721
0       700
Name: count, dtype: int64
   G       Date     Age   Tm  Home  Opp Unnamed: 7   GS     MP  FG  FGA  \
0  1 2003-10-29  18-303  CLE     0  SAC    L (-14)  1.0  42:50  12   20   
1  2 2003-10-30  18-304  CLE     0  PHO     L (-9)  1.0  40:21   8   17   
2  3 2003-11-01  18-306  CLE     0  POR    L (-19)  1.0  39:10   3   12   
3  4 2003-11-05  18-310  CLE     1  DEN     L (-4)  1.0  41:06   3   11   
4  5 2003-11-07  18-312  CLE     0  IND     L (-1)  1.0  43:44   8   18   

     FG%  3P  3PA  3P%  FT  FTA    FT%  ORB  DRB  TRB  AST  STL  BLK  TOV  PF  \
0  0.600   0    2  0.0   1    3  0.333    2    4    6    9    4    0    2   3   
1  0.471   1    5  0.2   4    7  0.571    2   10   12    8    1    0    7   1   
2  0.250   0    1  0.0   2    2  1.000    0    4    4    6    2    0    2   3   
3  0.273   0    2  0.0   1    1  1.000    2    9   11    7    2    3    2   1   
4  0.444   1    2  0.5   6    7  0.857    0    5    5    3    0    0    7   2   

   PTS 

# Step 4 — Remove unnecessary features

Drop columns that are not needed for this training dataset design.

Reason for each removed feature:

- `GS`: Starter flag has near-zero variance as LeBron starts almost every game, providing no predictive signal.
- `Tm_CLE`, `Tm_LAL`, `Tm_MIA`: Removed to prevent **Feature Drift** and chronological bias, as specific team history does not exist in every split.
- `G`: Game index is a chronological identifier and does not represent a physical gameplay signal.
- `Unnamed: 7`: Non-informative artifact column from the original data source export.
- `ORB`: Offensive rebounds are in-game outcomes that would cause **Data Leakage** if used to predict the same game.
- `DRB`: Defensive rebounds are excluded to prevent data leakage and keep the feature space focused on pre-game indicators.
- `TRB`: Total rebounds are redundant with the rolling average features and occur during the game, causing leakage.
- `AST`: Assist totals are in-game performance results and are excluded to avoid "look-ahead" bias.
- `STL`: Defensive steals are in-game outcomes; their predictive value is better captured by rolling averages.
- `BLK`: Blocks are excluded to prevent data leakage and focus solely on pre-game historical trends.
- `TOV`: Turnover counts are result-based stats that are unknown at the time of prediction.
- `PF`: Personal fouls are highly situational in-game events that do not serve as stable predictors.
- `GmSc`: Composite metric calculated after the game; removed to prevent direct data leakage.
- `+/-`: Team-based outcome stat that depends on other players' performance, causing instability and leakage.
- `FG`: Raw field goals made are removed to prevent leakage, as they directly reveal the final score.
- `3P`: Three-pointers made are results-based and removed to focus on volume-based attempts (3PA).
- `FT`: Free throws made are excluded to prevent the model from "cheating" using in-game success rates.

Why this is needed for training: Removing in-game outcomes prevents **Data Leakage**, while pruning redundant features reduces **Multicollinearity** and ensures the model relies on historical momentum rather than post-game summaries.

In [ ]:
# Remove the GS feature from the dataframe
df.drop(columns=['GS'], inplace=True)
df.drop(columns=['Tm'], inplace=True)
df.drop(columns=['G'], inplace=True)
df.drop(columns=['Unnamed: 7'], inplace=True)
df.drop(columns=['ORB'], inplace=True)
df.drop(columns=['DRB'], inplace=True)
df.drop(columns=['TRB'], inplace=True)
df.drop(columns=['AST'], inplace=True)
df.drop(columns=['STL'], inplace=True)
df.drop(columns=['BLK'], inplace=True)
df.drop(columns=['TOV'], inplace=True)
df.drop(columns=['PF'], inplace=True)
df.drop(columns=['GmSc'], inplace=True)
df.drop(columns=['+/-'], inplace=True)
df.drop(columns=['FG'], inplace=True)
df.drop(columns=['3P'], inplace=True)
df.drop(columns=['FT'], inplace=True)

df.drop(columns=['Opp'], inplace=True)


# Step 5 — Encode opponent team (`Opp`)

Apply one-hot encoding to `Opp`, then drop the original text column.

Why this is needed for training: most models require numeric inputs, and one-hot encoding preserves team identity without creating false numeric order.

In [15]:
# One-hot encode the Opp column (NBA opponent teams)
if 'Opp' in df.columns:
    opp_dummies = pd.get_dummies(df['Opp'], prefix='Opp', dtype=int)
    df = pd.concat([df.drop(columns=['Opp']), opp_dummies], axis=1)

    print("One-hot encoding complete.")
    print("New shape:", df.shape)
    print("\nSample of encoded team columns:")
    print(df.filter(like='Opp_').head())
else:
    print("'Opp' column not found (it may already be encoded).")

One-hot encoding complete.
New shape: (1421, 46)

Sample of encoded team columns:
   Opp_ATL  Opp_BOS  Opp_BRK  Opp_CHA  Opp_CHI  Opp_CHO  Opp_CLE  Opp_DAL  \
0        0        0        0        0        0        0        0        0   
1        0        0        0        0        0        0        0        0   
2        0        0        0        0        0        0        0        0   
3        0        0        0        0        0        0        0        0   
4        0        0        0        0        0        0        0        0   

   Opp_DEN  Opp_DET  Opp_GSW  Opp_HOU  Opp_IND  Opp_LAC  Opp_LAL  Opp_MEM  \
0        0        0        0        0        0        0        0        0   
1        0        0        0        0        0        0        0        0   
2        0        0        0        0        0        0        0        0   
3        1        0        0        0        0        0        0        0   
4        0        0        0        0        1        0        0      

# Step 6 — Convert `Age` and `MP` into numeric format

- `Age`: convert from `YY-DDD` to decimal years (`YY + DDD/365`).
- `MP`: convert from `MM:SS` to decimal minutes (`MM + SS/60`).

Why this is needed for training: numeric continuous values are directly usable by ML algorithms and avoid parsing strings during modeling.

In [16]:
# Convert Age from "YY-DDD" -> YY + DDD/365
if 'Age' in df.columns:
    age_parts = df['Age'].astype(str).str.extract(r'^\s*(\d+)-(\d+)\s*$')
    df['Age'] = pd.to_numeric(age_parts[0], errors='coerce') + (
        pd.to_numeric(age_parts[1], errors='coerce') / 365
    )

# Convert MP from "MM:SS" -> decimal minutes
if 'MP' in df.columns:
    mp_parts = df['MP'].astype(str).str.extract(r'^\s*(\d+):(\d+)\s*$')
    df['MP'] = pd.to_numeric(mp_parts[0], errors='coerce') + (
        pd.to_numeric(mp_parts[1], errors='coerce') / 60
    )

print(df[['Age', 'MP']].head())
print(df[['Age', 'MP']].dtypes)

         Age         MP
0  18.830137  42.833333
1  18.832877  40.350000
2  18.838356  39.166667
3  18.849315  41.100000
4  18.854795  43.733333
Age    float64
MP     float64
dtype: object


# Step 7 — Create the target label (`HSG`)

Create `HSG` from `PTS` using the threshold `PTS >= 25` (1 = high-scoring game, 0 = otherwise).

Why this is needed for training: this defines the supervised learning target the model will predict.

In [17]:
# Create target column: High Scoring Game (HSG) from PTS
if 'PTS' in df.columns:
    df['HSG'] = (df['PTS'] >= 25).astype(int)

    print(df[['PTS', 'HSG']].head())
    print("\nClass distribution:")
    print(df['HSG'].value_counts())

else:
    print("'PTS' column not found.")

   PTS  HSG
0   25    1
1   21    0
2    8    0
3    7    0
4   23    0

Class distribution:
HSG
1    895
0    526
Name: count, dtype: int64


# Step 8 — Handle missing percentage values

Fill missing values in `FG%`, `3P%`, and `FT%` with `0` for existing columns only.

Why this is needed for training: many models cannot train with `NaN` values, and this keeps the dataset fully numeric and complete.

In [18]:
cols_to_fill = ['FG%', '3P%', 'FT%']
existing_cols = [c for c in cols_to_fill if c in df.columns]

df[existing_cols] = df[existing_cols].fillna(0)

print("NaN values after fill:")
print(df[existing_cols].isna().sum())

NaN values after fill:
FG%    0
3P%    0
FT%    0
dtype: int64


# Step 9 — Preview the cleaned dataset

Expand display settings and inspect the dataframe to confirm transformations and column layout before export.

Why this is needed for training: quick visual validation helps catch schema mistakes before saving the final file.

In [19]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 10)

display(df)

,Date,Age,Home,MP,FGA,FG%,3PA,3P%,FTA,FT%,PTS,Opp_ATL,Opp_BOS,Opp_BRK,Opp_CHA,Opp_CHI,Opp_CHO,Opp_CLE,Opp_DAL,Opp_DEN,Opp_DET,Opp_GSW,Opp_HOU,Opp_IND,Opp_LAC,Opp_LAL,Opp_MEM,Opp_MIA,Opp_MIL,Opp_MIN,Opp_NJN,Opp_NOH,Opp_NOK,Opp_NOP,Opp_NYK,Opp_OKC,Opp_ORL,Opp_PHI,Opp_PHO,Opp_POR,Opp_SAC,Opp_SAS,Opp_SEA,Opp_TOR,Opp_UTA,Opp_WAS,HSG
0,2003-10-29,18.830137,0,42.833333,20,0.600,2,0.000,3,0.333,25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1
1,2003-10-30,18.832877,0,40.350000,17,0.471,5,0.200,7,0.571,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2,2003-11-01,18.838356,0,39.166667,12,0.250,1,0.000,2,1.000,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,2003-11-05,18.849315,1,41.100000,11,0.273,2,0.000,1,1.000,7,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2003-11-07,18.854795,0,43.733333,18,0.444,2,0.500,7,0.857,23,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1416,2023-04-02,38.254795,0,29.350000,18,0.444,7,0.143,1,1.000,18,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1417,2023-04-04,38.260274,0,38.466667,27,0.519,10,0.300,6,1.000,37,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
1418,2023-04-05,38.263014,0,35.100000,20,0.650,6,0.667,5,0.600,33,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1419,2023-04-07,38.268493,1,29.350000,19,0.316,7,0.429,2,0.500,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


# Step 10 — Verify final data types

Print full data types to ensure each feature is in the expected format (numeric, datetime, encoded integer columns).

Why this is needed for training: type consistency prevents downstream errors in model fitting and preprocessing pipelines.

In [20]:
print(df.dtypes.to_string())

Date       datetime64[us]
Age               float64
Home                int64
MP                float64
FGA                 int64
FG%               float64
3PA                 int64
3P%               float64
FTA                 int64
FT%               float64
PTS                 int64
Opp_ATL             int64
Opp_BOS             int64
Opp_BRK             int64
Opp_CHA             int64
Opp_CHI             int64
Opp_CHO             int64
Opp_CLE             int64
Opp_DAL             int64
Opp_DEN             int64
Opp_DET             int64
Opp_GSW             int64
Opp_HOU             int64
Opp_IND             int64
Opp_LAC             int64
Opp_LAL             int64
Opp_MEM             int64
Opp_MIA             int64
Opp_MIL             int64
Opp_MIN             int64
Opp_NJN             int64
Opp_NOH             int64
Opp_NOK             int64
Opp_NOP             int64
Opp_NYK             int64
Opp_OKC             int64
Opp_ORL             int64
Opp_PHI             int64
Opp_PHO     

# Step 11 — Export cleaned dataset

Save the cleaned output as `LJ_Dataset_NODERIV.csv`.

This file is the prepared baseline dataset ready for the next feature-engineering or modeling stage.

In [21]:
df.to_csv('data/processed/LJ_Dataset_NODERIV.csv', index=False)
print("Exported dataset to LJ_Dataset_NODERIV.csv")

Exported dataset to LJ_Dataset_NODERIV.csv
